<img src="logo.png" alt="Vegeta" width="240">

# Project Onager — Onager Manus, two manipulator pincers

The **Manus** is the Onager series' working unit: the Sentinel's wheel-leg chassis (notebook 20) without the
sensor turret, with two manipulator arms on the front of the roof. Each arm has four joints (a yaw pedestal,
shoulder, elbow, wrist) and ends in a **pincer**: two jaws on one pin driven by a ball screw, a hardened cutter
notch 40 mm from the pin (it cuts wire) and serrated jaws with **hooked tips** (they lift a log).

The job it is sized for (inputs): a track blocked by a **high-tensile fence wire** at 1.15 m (Ø 3.15 mm, 1200 MPa)
and, further on, a **14 kg log** lying across it. The Manus cuts the wire, drives through, lifts the log off the
track, puts it down beside it and drives on. The notebook checks the design end to end:

```
CAD (Dedalus): chassis + two arms + pincers ─► masses, CG, the arm's reach
the arms: workspace (inverse kinematics), joint torques holding the log at reach, the modules (actuators.py)
the pincer: cutting force along the jaw vs the wire classes; why straight jaws drop a round log and hooks do not
FEA (Talos): the jaw at the cutting load, the upper arm with the log; the upper arm's modes
MuJoCo (ChironLab, Chiron props, welds, hooks): the mission — the wire parts only if the jaws squeeze it hard
enough, the log stays in the jaws only if the contacts hold it — with a movie
```

In [ ]:
import json, math, shutil, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from vegeta import dedalus, talos, chiron
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
from vegeta.chiron import viz as cviz
sys.path.insert(0, "designs")
import actuators as act
import onager_robot as orb, onager_controller as oc
import onager_manus_robot as omr, onager_manus_scenario as oms

RUNS = Path("_runs/onager_manus"); shutil.rmtree(RUNS, ignore_errors=True); RUNS.mkdir(parents=True)
OUT = Path("output/22_onager_manus"); OUT.mkdir(parents=True, exist_ok=True)
for f in ("onager.py", "onager_manus.py"):
    shutil.copy(Path("designs") / f, RUNS / f)
sys.path.insert(0, str(RUNS))
design = dedalus.load_design(f"{RUNS / 'onager_manus.py'}:OnagerManus")
G = 9.81
JOB = pd.Series({"wire": f"Ø {oms.WIRE['diameter_m'] * 1000:.2f} mm, {oms.WIRE['uts_MPa']:.0f} MPa (high-tensile fence wire)",
                 "cutting force [N]": round(oms.F_CUT), "wire height [m]": 1.15,
                 "log": f"spruce Ø {oms.LOG['diameter_m'] * 1000:.0f} mm × {oms.LOG['length_m']:.1f} m", "log mass [kg]": round(oms.LOG["mass_kg"], 1)},
                name="Onager Manus — the job (inputs)")
display(JOB.to_frame())
pd.DataFrame(design.params.table()).set_index("name").loc[list(omr.MANUS)]

## 1. The machine

In [ ]:
p = design.resolve()
parts = {k: design.generate(part=k) for k in ("robot", "upper_arm", "forearm", "jaw")}
cad = {k: g.export(RUNS / "cad" / k, stl_tolerance=0.1) for k, g in parts.items()}
ag = omr.arm_geometry()
print(f"shoulder pitch axes {ag['z'] + orb.geometry(omr.design_params())['hull_z']:.2f} m above the ground, {ag['x']:.2f} m ahead of the hull centre, ±{ag['y']:.2f} m | "
      f"reach (shoulder to jaw tips) {ag['Lu'] + ag['Lf'] + ag['Lp'] + ag['Lj']:.2f} m")
dviz.show(dviz.plot3d(parts["robot"]))

In [ ]:
dviz.show(dviz.plot3d(parts["jaw"]))

In [ ]:
cad_num = omr.cad_numbers(omr.design_params(), recompute=True)
assert max(abs(cad_num[k] / omr.CAD[k] - 1) for k in omr.CAD) < 0.01
budget = omr.mass_budget(cad=cad_num)
mass = pd.DataFrame({"mass [kg]": budget}); M = mass["mass [kg]"].sum(); mass.loc["TOTAL"] = M
arm = pd.DataFrame({"one arm [kg]": omr.arm_masses(cad=cad_num)}); arm.loc["TOTAL"] = arm["one arm [kg]"].sum()
robot = omr.manus(cad=cad_num)
lab_flat = omr.manus_lab(chiron.Flat(), robot=robot)
ep_stand = lab_flat.run(oc.Stand(), duration=2.0, rules=None, settle=0.0)
fz = np.asarray(ep_stand.log["foot_force"])[-1, :, 2]
com = np.asarray(ep_stand.log["com"])[-1]
print(f"Manus {M:.0f} kg (Sentinel 408 kg; two arms {2 * arm.loc['TOTAL'].iloc[0]:.0f} kg) | CG {com[0] * 1000:+.0f} mm, {com[2]:.2f} m up | wheels {fz.round(0)} N")
display(arm.round(2))
mass.round(1)

## 2. The arms: workspace and joint torques

The pincer's pin is placed by the arm's inverse kinematics (`onager_manus_robot.arm_ik`: yaw from the target's
bearing, a two-link elbow-up solution in the arm's plane, the wrist setting the pincer's elevation). The maps show
where in the arm's plane (forward of and below the shoulder) the pin can go with the pincer horizontal (cutting a
wire) and pointing down (taking a log), with the robot standing and crouched 0.30 m on its legs. Holding the log
at a given pin position, the static joint torques follow from the masses of §1.

In [ ]:
rs, zs = np.linspace(0.0, 1.5, 151), np.linspace(-1.6, 0.8, 241)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)
x_sh = ag["x"]
for ax, (elev, label) in zip(axes, ((0.0, "pincer horizontal (wire)"), (-math.pi / 2, "pincer down (log)"))):
    ok = np.array([[omr.arm_ik((x_sh + r, ag["y"], ag["z"] + z), elev, "L") is not None for r in rs] for z in zs])
    ax.contourf(rs, zs, ok, levels=[0.5, 1.5], colors=["#90caf9"])
    hull_top = -ag["ped_h"]
    ax.axhline(hull_top, color="#555", lw=0.8); ax.text(0.02, hull_top + 0.02, "roof", fontsize=8)
    for crouch, ls in ((0.0, "-"), (0.30, "--")):
        ground = -(ag["z"] + orb.geometry(omr.design_params())["hull_z"] - crouch)
        ax.axhline(ground, color="#6d4c41", ls=ls, lw=1); ax.text(1.0, ground + 0.02, f"ground, crouch {crouch:.2f} m", fontsize=8, color="#6d4c41")
    ax.set(title=label, xlabel="forward of the shoulder [m]"); ax.grid(alpha=0.3)
axes[0].set_ylabel("above the shoulder [m]")
z_wire_rel = 1.15 - (ag["z"] + orb.geometry(omr.design_params())["hull_z"])
axes[0].plot(0.85, z_wire_rel, "r*", ms=12); axes[0].annotate("the cut", (0.85, z_wire_rel), xytext=(5, 5), textcoords="offset points")
m_arm = omr.arm_masses(cad=cad_num)
def joint_torques(r, z_pin, elev, load_kg):
    """Static shoulder, elbow, wrist torques [N m] with the pin at (r forward, z_pin above the shoulder), the load at the pin."""
    q = omr.arm_ik((x_sh + r, ag["y"], ag["z"] + z_pin), elev, "L")
    if q is None:
        return (np.nan,) * 3
    _, sh, el, wr = q
    p1, p2, p3 = -sh, -(sh + el), -(sh + el + wr)
    elbow = ag["Lu"] * math.cos(p1); wrist = elbow + ag["Lf"] * math.cos(p2); pin = wrist + ag["Lp"] * math.cos(p3)
    jaw_c = pin + 0.1 * math.cos(p3)
    items = [(m_arm["upper arm (Al 6082, CAD)"], ag["Lu"] / 2 * math.cos(p1)), (m_arm["elbow drive (harmonic 150 Nm, brake)"], elbow),
             (m_arm["forearm (Al 6082, CAD)"], elbow + ag["Lf"] / 2 * math.cos(p2)), (m_arm["wrist drive (harmonic 60 Nm, brake)"], wrist),
             (m_arm["palm (wrist housing, screw mount)"] + m_arm["jaw drive (jaw screw 6 kN)"], wrist + ag["Lp"] / 2 * math.cos(p3)),
             (m_arm["jaws 2x (tool steel, CAD)"], jaw_c), (load_kg, jaw_c)]
    tau_sh = G * sum(mi * xi for mi, xi in items)
    tau_el = G * sum(mi * (xi - elbow) for mi, xi in items[2:])
    tau_wr = G * sum(mi * (xi - wrist) for mi, xi in items[4:])
    return abs(tau_sh), abs(tau_el), abs(tau_wr)
reach = np.linspace(0.3, 1.0, 15)
rows = {}
for r in reach:
    rows[f"{r:.2f} m"] = dict(zip(("shoulder", "elbow", "wrist"), joint_torques(r, -0.75, -math.pi / 2, oms.LOG["mass_kg"])))
tq = pd.DataFrame(rows).T
stall = {j: act.get(omr.ARM_ACTUATORS[j]).stall_Nm for j in ("shoulder", "elbow", "wrist")}
fig, ax = plt.subplots(figsize=(8, 4))
for j, col in zip(stall, ("#1565c0", "#2e7d32", "#ef6c00")):
    ax.plot(reach, tq[j] / stall[j], color=col, label=f"{j} ({omr.ARM_ACTUATORS[j]})")
ax.axhline(1, color="k", lw=0.8); ax.axhline(1 / 1.5, color="k", ls=":", lw=0.8, label="SF 1.5")
ax.set(xlabel="pin forward of the shoulder [m] (pin 0.75 m below it, pincer down)", ylabel="static torque / stall", title=f"holding the {oms.LOG['mass_kg']:.0f} kg log"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
heaviest = min(((stall["shoulder"] / 1.5 - joint_torques(0.7, -0.75, -math.pi / 2, 0)[0]) / (G * (0.7 + 0.1)), 1e9), key=float)
print(f"the heaviest log at 0.7 m reach with SF 1.5 on the shoulder: ≈ {heaviest:.0f} kg (shoulder module {stall['shoulder']:.0f} N·m)")
tq.round(1).T

## 3. The pincer: cutting and gripping

**Cutting.** The jaw drive (a 6 kN ball screw on a 60 mm lever: 360 N·m at the pin, `jaw screw 6 kN` in the
catalogue) squeezes a wire with `τ / x` at its distance `x` from the pin: the notch is at 40 mm, 9 kN. A wire needs
`0.8 × UTS × area` (the shear strength of drawn steel wire, an input). The chart is the pincer's cutting envelope.

**Gripping.** Straight jaws closing on a round log from above touch it above its middle: the contact pushes the log
**down** with `N sin α` (α the jaw's angle at contact, `sin α = r / depth`) and friction holds it up with
`μ N cos α`. They hold only if `μ > tan α` and the squeeze exceeds `m g / (2 (μ cos α − sin α))` — and the same
wedge pushes the arm **up** with `2 N sin α`, which the shoulder must resist. That is what the first MuJoCo runs
showed: the log slipped out and the arm was pushed up 6 cm. The hooked tips close under the log's middle and carry
it on their faces: no friction needed, no wedge on the arm.

In [ ]:
JAW = act.get(omr.ARM_ACTUATORS["jaw"])
xs = np.linspace(0.02, ag["Lj"], 100)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
axes[0].plot(xs * 1000, JAW.stall_Nm / xs / 1000, "k", lw=2, label="squeeze at stall, τ / x")
WIRES = {"fence, mild Ø 2.5 (400 MPa)": (2.5, 400), "barbed, Ø 2.5 (700 MPa)": (2.5, 700), "fence, high-tensile Ø 3.15 (1200 MPa)": (3.15, 1200),
         "high-tensile Ø 4.0 (1200 MPa)": (4.0, 1200), "rebar Ø 6 (550 MPa)": (6.0, 550)}
cut_tab = {}
for (name, (dw, uts)), col in zip(WIRES.items(), ("#90caf9", "#64b5f6", "#c62828", "#ef6c00", "#6d4c41")):
    F = 0.8 * uts * math.pi / 4 * dw ** 2
    axes[0].axhline(F / 1000, color=col, ls="--", lw=1, label=f"{name}: {F / 1000:.1f} kN")
    x_max = JAW.stall_Nm / F
    cut_tab[name] = {"cutting force [kN]": F / 1000, "cuts up to x [mm] from the pin": x_max * 1000, "in the notch (40 mm)": x_max >= ag["notch"]}
axes[0].axvline(ag["notch"] * 1000, color="k", ls=":"); axes[0].text(ag["notch"] * 1000, 30, " notch", fontsize=8)
axes[0].set(xlabel="distance from the jaw pin [mm]", ylabel="kN", ylim=(0, 40), title="cutting envelope"); axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3)
mu, r_log, m_log = oms.LOG["mu_on_jaws"], oms.LOG["diameter_m"] / 2, oms.LOG["mass_kg"]
depths = np.linspace(0.09, 0.22, 60)
alpha = np.arcsin(r_log / depths)
margin = mu * np.cos(alpha) - np.sin(alpha)
N_need = np.where(margin > 0, m_log * G / (2 * np.maximum(margin, 1e-9)), np.nan)
axes[1].plot(depths * 1000, N_need, label="squeeze per jaw needed [N] (straight jaws)")
axes[1].plot(depths * 1000, 2 * N_need * np.sin(alpha), label="upward push on the arm [N]")
axes[1].axhline(JAW.stall_Nm / 0.185, color="k", ls=":", label="jaw drive at stall, contact 185 mm out")
axes[1].set(xlabel="log centre below the jaw pin [mm]", ylabel="N", yscale="log", title=f"straight jaws on a {m_log:.0f} kg log, μ = {mu}"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3, which="both")
pd.DataFrame(cut_tab).T.round(2)

## 4. FEA (Talos): the jaw at the cutting load, the upper arm with the log

**Jaw** (tool steel, its own frame): the pin bore held, the cutting reaction on the notch's faces — the drive's
stall squeeze at the notch, 9 kN, the worst case the jaw sees (the wire's 7.5 kN is less). **Upper arm** (Al
6082-T6 rectangular tube): the shoulder bore held, at the elbow bore the weight of everything outboard — forearm,
wrist, pincer and the log — with a dynamic factor 2 (lifting and swinging), arm horizontal. Then the upper arm's
modes with the outboard mass lumped at the elbow.

In [ ]:
TOOL = talos.Material("tool steel jaw (42CrMo4 QT)", youngs_modulus=210000.0, poissons_ratio=0.30, density=7.85e-9, yield_strength=900.0, source="EN 10083-3")
AL6082 = talos.Material("Al 6082-T6 tube", youngs_modulus=70000.0, poissons_ratio=0.33, density=2.7e-9, yield_strength=260.0, source="EN 755-2")
jt, jd, nx_, nd_ = p["jaw_thickness"], p["jaw_depth"], p["notch_x"], p["notch_depth"]
rp = p["jaw_pin_diameter"] / 2 + 0.5
JAW_REGIONS = [talos.SurfacesInBox("pin", (-rp, -jt, jd / 2 - rp, rp, jt, jd / 2 + rp)),
               talos.SurfacesInBox("notch", (nx_ - nd_ - 0.5, -jt, -0.5, nx_ + nd_ + 0.5, jt, nd_ + 0.5))]
F_notch = JAW.stall_Nm / ag["notch"]
jm = talos.StructuralModel(cad["jaw"].artifacts["step"], "mm-N-MPa", TOOL, JAW_REGIONS, [talos.FixedSupport("pin")], [talos.Force("notch", fz=F_notch)],
                           talos.MeshSettings(element_size=2.0, min_element_size=0.6), name="jaw_cut")
jm.mesh(RUNS / "jaw_cut", progress=True); jaw_res = jm.solve(RUNS / "jaw_cut")
Lu, uw, ud = p["upper_arm_length"], p["upper_arm_width"], p["upper_arm_depth"]
rb = p["arm_boss_diameter"] / 2 + 0.5
ARM_REGIONS = [talos.SurfacesInBox("shoulder", (-rb, -uw, -rb, rb, uw, rb)), talos.SurfacesInBox("elbow", (Lu - rb, -uw, -rb, Lu + rb, uw, rb))]
outboard = (m_arm["forearm (Al 6082, CAD)"] + m_arm["wrist drive (harmonic 60 Nm, brake)"] + m_arm["palm (wrist housing, screw mount)"]
            + m_arm["jaw drive (jaw screw 6 kN)"] + m_arm["jaws 2x (tool steel, CAD)"] + m_arm["elbow drive (harmonic 150 Nm, brake)"] + m_log)
lever_out = 0.55                                         # outboard masses' centroid beyond the elbow [m] (arm horizontal)
F_elbow = outboard * G * 2
M_elbow = F_elbow * lever_out
am = talos.StructuralModel(cad["upper_arm"].artifacts["step"], "mm-N-MPa", AL6082, ARM_REGIONS, [talos.FixedSupport("shoulder")],
                           [talos.Force("elbow", fz=-F_elbow)], talos.MeshSettings(element_size=6.0, min_element_size=2.0), name="arm_log")
am.mesh(RUNS / "arm_log"); arm_res = am.solve(RUNS / "arm_log")
# the outboard moment is the larger part of the upper arm's load; a tube in bending by hand for the combined case:
I_tube = (uw * ud ** 3 - (uw - 2 * p["arm_wall"]) * (ud - 2 * p["arm_wall"]) ** 3) / 12
sigma_hand = (F_elbow * Lu + M_elbow * 1000) * (ud / 2) / I_tube
fea = pd.DataFrame({
    "jaw at the stall squeeze (notch)": {"load": f"{F_notch / 1000:.1f} kN", "max von Mises [MPa]": jaw_res.metrics.get("max_von_mises"), "SF yield": jaw_res.metrics.get("safety_factor_yield"), "max displacement [mm]": jaw_res.metrics.get("max_displacement")},
    "upper arm, outboard weight × 2 (FEA: force at the elbow)": {"load": f"{F_elbow:.0f} N", "max von Mises [MPa]": arm_res.metrics.get("max_von_mises"), "SF yield": arm_res.metrics.get("safety_factor_yield"), "max displacement [mm]": arm_res.metrics.get("max_displacement")},
    "upper arm, + the outboard moment (hand, tube bending)": {"load": f"{F_elbow:.0f} N + {M_elbow:.0f} N·m", "max von Mises [MPa]": sigma_hand, "SF yield": AL6082.yield_strength / sigma_hand, "max displacement [mm]": float("nan")},
}).T
fea.round(2)

In [ ]:
tviz.show(tviz.plot_results(jaw_res, field="von_mises"))

In [ ]:
modal = talos.StructuralModel(cad["upper_arm"].artifacts["step"], "mm-N-MPa", AL6082, ARM_REGIONS, [talos.FixedSupport("shoulder")], [],
                              talos.MeshSettings(element_size=8.0, min_element_size=2.5), name="arm_modal", masses=[talos.PointMass("elbow", outboard * 1e-3)])
modal.mesh(RUNS / "arm_modal"); arm_modes = modal.solve_modes(RUNS / "arm_modal", n_modes=4)
f_arm = arm_modes.metrics["frequencies_hz"]
kp_sh = omr.ARM_GAINS["shoulder"][0]
I_out = outboard * (Lu / 1000 + lever_out) ** 2 + m_arm["upper arm (Al 6082, CAD)"] * (Lu / 1000) ** 2 / 3 + omr.ARM_GAINS["shoulder"][2]
f_servo = math.sqrt(kp_sh / I_out) / (2 * math.pi)
print(f"upper arm with {outboard:.1f} kg at the elbow: {[round(f, 1) for f in f_arm]} Hz | the shoulder servo's own mode (kp {kp_sh:.0f} N·m/rad on {I_out:.1f} kg·m²): {f_servo:.1f} Hz — the servo, not the tube, sets the arm's bandwidth")
tviz.show(tviz.plot_mode(arm_modes, mode=1))

## 5. The mission in MuJoCo (ChironLab)

The Manus as a Chiron robot (`designs/onager_manus_robot.py`: the Sentinel chassis's servos and wheels, 12 arm
and jaw servos on the catalogue's torque–speed lines, the hooked jaws as boxes). The scene
(`designs/onager_manus_scenario.py`) uses Chiron's scenery: two posts and the wire as two rigid halves, each hinged
at its post and held to the other by a **weld** exactly where the cutting pincer meets it; a scene **hook** releases
the weld only when **both** jaws squeeze the wire with at least the cutting force for 50 ms — MuJoCo's contact
forces decide; the log is a free **prop** on the gravel with bark-like rolling resistance. The mission
(`designs/onager_manus_controller.py`, a `PhasedMission` with arms) knows where the wire runs and reads the log's
pose when it reaches for it (ideal perception).

In [ ]:
scene = oms.Scene()
lab = oms.make_lab(scene, robot=robot, log_geoms=True)
ep = oms.run(lab, scene, duration=66.0)
ep.save(RUNS / "mission.npz")
ts = oms.timeseries(ep)
print(f"mission finished: {ep.log['mission_finished']} | events: {ep.log['events']}")
print(f"log: lifted to {ts.log_z.max():.2f} m, put down at y = {ts.log_y.iloc[-1]:+.2f} m (the wheels run at ±{orb.geometry(omr.design_params())['y_wheel'] + orb.geometry(omr.design_params())['w_wheel'] / 2:.2f} m) | robot at x = {ts.x.iloc[-1]:.2f} m")
phases = oms.phase_table(ep)
phases.round(2)

In [ ]:
h = np.asarray(ep.log["cutter_history"])
fig, axes = plt.subplots(2, 2, figsize=(15, 8))
ax = axes[0, 0]
if len(h):
    ax.plot(h[:, 0], h[:, 1] / 1000, label="upper jaw on the wire"); ax.plot(h[:, 0], h[:, 2] / 1000, label="lower jaw on the wire")
ax.axhline(oms.F_CUT / 1000, color="#c62828", ls="--", label=f"cutting force {oms.F_CUT / 1000:.1f} kN")
if ep.log["cut_at"] is not None:
    ax.axvline(ep.log["cut_at"], color="k", lw=0.8); ax.text(ep.log["cut_at"], 1, " the wire parts", fontsize=8)
ax.set(xlabel="time [s]", ylabel="kN", title="the squeeze on the wire"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax = axes[0, 1]
ax.plot(ts.t, ts.wire_a_deg, label="wire half a (short)"); ax.plot(ts.t, ts.wire_b_deg, label="wire half b (long)")
ax.set(xlabel="time [s]", ylabel="deg about its post", title="the parted wire swings down"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax = axes[1, 0]
ax.plot(ts.t, ts.log_z, label="log height"); ax.plot(ts.t, ts.log_y, label="log lateral"); ax.plot(ts.t, ts.x / 20, label="hull x / 20")
ax.set(xlabel="time [s]", ylabel="m", title="the log"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax = axes[1, 1]
for side, ls in (("L", "-"), ("R", "--")):
    for j, col in zip(("shoulder", "elbow", "wrist", "jaw_upper"), ("#1565c0", "#2e7d32", "#ef6c00", "#6d4c41")):
        key = "jaw" if j.startswith("jaw") else j
        ax.plot(ts.t, ts[f"tau_{side}_{j}"].abs() / act.get(omr.ARM_ACTUATORS[key]).stall_Nm, ls, color=col, lw=0.8, label=f"{side} {j}")
ax.axhline(1, color="k", lw=0.8); ax.set(xlabel="time [s]", ylabel="|τ| / stall", title="arm and jaw drives"); ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.3)
for a in axes.flat:
    for t0, name, note in ep.log["mission"]:
        if note == "start": a.axvline(t0, color="#ddd", lw=0.5, zorder=0)
fig.tight_layout()

In [ ]:
dt = float(ep.log["t"][1] - ep.log["t"][0]); every = max(1, int(round(1 / (25 * dt))))
imgs = cviz.frames(ep, camera="follow", every=every, size=(960, 540))
movie = cviz.to_video(imgs, OUT / "onager_manus_tasks.mp4", fps=25)
print(f"movie: {movie} ({len(imgs)} frames; the wire is drawn 4× thicker)")
starts = {}
for t0, name, note in ep.log["mission"]:
    if note == "start": starts.setdefault(name, t0)
fig, axes = plt.subplots(2, 3, figsize=(18, 6.5))
for ax, (name, off) in zip(axes.flat, (("R arm: notch onto the wire", 1.8), ("open the jaws", 0.8), ("drive through to the log", 1.0),
                                       ("lift", 2.0), ("swing over the side", 2.5), ("drive on", 3.0))):
    j = min(len(imgs) - 1, int(round((starts.get(name, 0.0) + off) / (every * dt))))
    ax.imshow(imgs[j]); ax.set_axis_off(); ax.set_title(f"{name} (t = {j * every * dt:.1f} s)", fontsize=9)
fig.tight_layout()

## 6. Export

In [ ]:
doc = {"source": "22_onager_manus", "design": {"file": "designs/onager_manus.py", "parameters": p}, "job": JOB.to_dict(),
       "mass_kg": M, "mass_budget_kg": budget, "arm_kg": arm["one arm [kg]"].to_dict(), "arm_torques_holding_log": tq.to_dict(orient="index"),
       "cutting": cut_tab, "fea": fea.to_dict(orient="index"), "upper_arm_modes_hz": f_arm, "shoulder_servo_mode_hz": f_servo,
       "mission": {"finished": ep.log["mission_finished"], "events": ep.log["events"], "phases": phases.to_dict(orient="index"),
                   "log_max_height_m": float(ts.log_z.max()), "log_final_y_m": float(ts.log_y.iloc[-1]), "movie": str(movie)}}
(RUNS / "onager_manus.json").write_text(json.dumps(doc, indent=2, default=float))
print("written:", RUNS / "onager_manus.json")

**What the numbers say** — read them above; the ones that set the design: the cutting envelope (the notch cuts the
high-tensile wire with ~20 % margin at stall; 4 mm high-tensile wire or rebar needs the wire deeper in the notch or
a stronger screw); the gripping analysis (straight jaws would need a crushing squeeze and push the arm up — the
hooks are the design decision); the arm torques holding the log at reach against the modules; the jaw and arm
FEA; and in §5 the squeeze trace (both jaws past the cutting force before the wire parts), the log's height and
where it was put down.

**Next steps an engineer would take:** a jaw linkage model (one screw, both jaws) and the notch's real V geometry
(the wire seats itself; here it is held at 40 mm by the approach); a wrist roll joint (a log that does not lie
square to the arm); a cutting-force test on real wire samples (the 0.8 × UTS is a handbook ratio); the arms'
reaction on the chassis while the robot drives with a load (the second arm as a counterweight); and grasping with
perception noise instead of the simulated pose.